# Exercise 3 + Bonus 4 scaffolding

Implement the stubs. Tests are provided (synthetic tensors / CPU where possible).


## Prerequisite: Exercise 2

Use the finished Exercise 2 notebook numbers (kNN, silhouette, recon) as the baseline.


In [2]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True NVIDIA A100-SXM4-80GB


## Setup

Small in-notebook test harness (`@test` + `run()`).


In [3]:
import math
import numpy as np
import torch
from torch import nn

LATENT_DIM = 8      # small on purpose: tests stay fast
BATCH = 4

_TESTS = {}


def test(fn):
    """Register a test function so `run()` can find it."""
    _TESTS[fn.__name__] = fn
    return fn


def close(a, b, rel=1e-5, abs_tol=1e-8):
    """Float comparison. Stands in for pytest.approx."""
    return math.isclose(float(a), float(b), rel_tol=rel, abs_tol=abs_tol)


def run(prefix=""):
    """Run every registered test whose name starts with `prefix`."""
    names = sorted(n for n in _TESTS if n.startswith(prefix))
    if not names:
        print(f"no tests matching {prefix!r}")
        return False
    failures = []
    for name in names:
        try:
            _TESTS[name]()
        except Exception as exc:
            failures.append((name, f"{type(exc).__name__}: {exc}"))
    passed = len(names) - len(failures)
    print(f"{passed}/{len(names)} passed")
    for name, err in failures:
        print(f"  FAIL  {name}")
        print(f"        {err}")
    return not failures


device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "cpu")

device: cuda | NVIDIA A100-SXM4-80GB


## Data

Reuse the Exercise 2 split (seed 42, 80/20). Point `BRISC_ROOT` at the dataset.


In [4]:
from brisc import build, device, transform, ROOT, CLASS_TRAIN

classification_data_transform, train_data, val_data, train_loader, val_loader, encode_loader = build()

print("root   ", ROOT)
print("classes", classification_data_transform.classes)
print("images ", len(classification_data_transform),
      "->", len(train_data), "train /", len(val_data), "val")
print("device ", device)


KeyboardInterrupt


KeyboardInterrupt



In [ ]:
# Loaders come from cell 6 (brisc.build). Same seed 42 / 80/20 as Exercise 2.
print("train_loader", len(train_loader.dataset), "val_loader", len(val_loader.dataset))
print("CLASS_TRAIN", CLASS_TRAIN)



## Task 3 — VAE vs AE

AE codes are unconstrained points. VAE adds a posterior + KL so the latent is a space you can sample and interpolate.


### Stubs

Fill the imports/implementations below; then run the matching tests.


In [ ]:
from exercise3_vae import ConvVAE



In [ ]:
from exercise3_vae import kl_divergence, vae_loss



### Tests: construction and encoding

In [ ]:
@test
def test_construct_records_latent_dim():
    assert ConvVAE(latent_dim=2).latent_dim == 2
    assert ConvVAE(latent_dim=128).latent_dim == 128


@test
def test_construct_has_trainable_parameters():
    model = ConvVAE(latent_dim=LATENT_DIM)
    assert any(p.requires_grad for p in model.parameters())


@test
def test_encode_returns_two_tensors():
    out = ConvVAE(LATENT_DIM).encode(torch.rand(BATCH, 1, 128, 128))
    assert isinstance(out, tuple) and len(out) == 2


@test
def test_encode_shapes():
    mu, logvar = ConvVAE(LATENT_DIM).encode(torch.rand(BATCH, 1, 128, 128))
    assert mu.shape == (BATCH, LATENT_DIM), mu.shape
    assert logvar.shape == (BATCH, LATENT_DIM), logvar.shape


@test
def test_encode_mu_and_logvar_are_distinct():
    """One linear layer reused for both would tie them together."""
    mu, logvar = ConvVAE(LATENT_DIM).encode(torch.rand(BATCH, 1, 128, 128))
    assert not torch.allclose(mu, logvar)


@test
def test_encode_different_inputs_differ():
    model = ConvVAE(LATENT_DIM)
    a = model.encode(torch.zeros(1, 1, 128, 128))[0]
    b = model.encode(torch.ones(1, 1, 128, 128))[0]
    assert not torch.allclose(a, b)


run("test_construct") and run("test_encode")

### Reparameterization tests

Check shape, noise scale, and that gradients reach both μ and log σ².


In [ ]:
@test
def test_reparam_shape_matches_mu():
    model = ConvVAE(LATENT_DIM)
    mu = torch.zeros(BATCH, LATENT_DIM)
    assert model.reparameterize(mu, torch.zeros_like(mu)).shape == mu.shape


@test
def test_reparam_is_stochastic():
    """Two calls with the same arguments must not agree."""
    model = ConvVAE(LATENT_DIM)
    mu = torch.zeros(BATCH, LATENT_DIM)
    logvar = torch.zeros(BATCH, LATENT_DIM)
    assert not torch.allclose(
        model.reparameterize(mu, logvar), model.reparameterize(mu, logvar)
    )


@test
def test_reparam_collapses_to_mu_when_variance_vanishes():
    """With sigma effectively zero the sample must be mu itself."""
    model = ConvVAE(LATENT_DIM)
    mu = torch.randn(BATCH, LATENT_DIM)
    logvar = torch.full((BATCH, LATENT_DIM), -50.0)
    assert torch.allclose(model.reparameterize(mu, logvar), mu, atol=1e-4)


@test
def test_reparam_empirical_moments():
    """Draw many samples, check mean and standard deviation."""
    torch.manual_seed(0)
    model = ConvVAE(LATENT_DIM)
    n = 20000
    mu = torch.full((n, 1), 3.0)
    logvar = torch.full((n, 1), math.log(4.0))   # variance 4, sigma 2
    z = model.reparameterize(mu, logvar)
    assert close(z.mean(), 3.0, rel=0, abs_tol=0.05), z.mean().item()
    assert close(z.std(), 2.0, rel=0, abs_tol=0.05), z.std().item()


@test
def test_reparam_gradients_reach_both_arguments():
    """The trick itself. Plain sampling breaks this."""
    model = ConvVAE(LATENT_DIM)
    mu = torch.zeros(BATCH, LATENT_DIM, requires_grad=True)
    logvar = torch.zeros(BATCH, LATENT_DIM, requires_grad=True)
    model.reparameterize(mu, logvar).sum().backward()
    assert mu.grad is not None, "no gradient on mu"
    assert logvar.grad is not None, "no gradient on logvar"
    assert torch.isfinite(mu.grad).all()
    assert torch.isfinite(logvar.grad).all()


@test
def test_reparam_preserves_dtype_and_device():
    model = ConvVAE(LATENT_DIM)
    mu = torch.zeros(BATCH, LATENT_DIM, dtype=torch.float64)
    z = model.reparameterize(mu, torch.zeros_like(mu))
    assert z.dtype == mu.dtype
    assert z.device == mu.device


run("test_reparam")

### Decode / forward tests

Decoder must accept a prior sample with no input image.


In [ ]:
@test
def test_decode_shape():
    model = ConvVAE(LATENT_DIM)
    assert model.decode(torch.randn(BATCH, LATENT_DIM)).shape == (BATCH, 1, 128, 128)


@test
def test_decode_output_is_bounded():
    """The decoder ends in a sigmoid, matching the [0, 1] input range."""
    out = ConvVAE(LATENT_DIM).decode(torch.randn(BATCH, LATENT_DIM) * 10)
    assert out.min() >= 0.0 and out.max() <= 1.0


@test
def test_decode_accepts_a_prior_sample():
    out = ConvVAE(LATENT_DIM).decode(torch.randn(1, LATENT_DIM))
    assert out.shape == (1, 1, 128, 128)
    assert torch.isfinite(out).all()


@test
def test_forward_returns_three_values():
    out = ConvVAE(LATENT_DIM)(torch.rand(BATCH, 1, 128, 128))
    assert isinstance(out, tuple) and len(out) == 3


@test
def test_forward_shapes():
    x = torch.rand(BATCH, 1, 128, 128)
    x_hat, mu, logvar = ConvVAE(LATENT_DIM)(x)
    assert x_hat.shape == x.shape
    assert mu.shape == (BATCH, LATENT_DIM)
    assert logvar.shape == (BATCH, LATENT_DIM)


@test
def test_forward_reconstruction_is_bounded():
    x_hat, _, _ = ConvVAE(LATENT_DIM)(torch.rand(BATCH, 1, 128, 128))
    assert x_hat.min() >= 0.0 and x_hat.max() <= 1.0


@test
def test_forward_gradients_reach_the_encoder():
    """Sample without the trick and the encoder gets no gradient."""
    model = ConvVAE(LATENT_DIM)
    model(torch.rand(BATCH, 1, 128, 128))[0].sum().backward()
    grads = [p.grad for p in model.parameters() if p.grad is not None]
    assert grads, "no parameter received a gradient"
    assert any(g.abs().sum() > 0 for g in grads)


run("test_decode") and run("test_forward")

### KL tests

Closed-form KL to N(0,I). Includes a couple of hand-checkable scalars.


In [ ]:
@test
def test_kl_returns_one_value_per_sample():
    out = kl_divergence(torch.zeros(BATCH, LATENT_DIM), torch.zeros(BATCH, LATENT_DIM))
    assert out.shape == (BATCH,), out.shape


@test
def test_kl_zero_at_the_prior():
    """mu = 0 and variance = 1 IS the prior, so divergence is 0."""
    out = kl_divergence(torch.zeros(BATCH, LATENT_DIM), torch.zeros(BATCH, LATENT_DIM))
    assert torch.allclose(out, torch.zeros(BATCH), atol=1e-6), out


@test
def test_kl_never_negative():
    torch.manual_seed(0)
    out = kl_divergence(torch.randn(64, LATENT_DIM) * 3, torch.randn(64, LATENT_DIM) * 2)
    assert (out >= -1e-6).all(), out.min().item()


@test
def test_kl_shifted_mean_hand_computed():
    """One dimension, mu = 1, variance = 1. Expected 0.5."""
    out = kl_divergence(torch.tensor([[1.0]]), torch.tensor([[0.0]]))
    assert close(out.item(), 0.5, rel=0, abs_tol=1e-5), out.item()


@test
def test_kl_widened_variance_hand_computed():
    """One dimension, mu = 0, logvar = 1. Expected 0.3591409."""
    out = kl_divergence(torch.tensor([[0.0]]), torch.tensor([[1.0]]))
    assert close(out.item(), 0.5 * (math.e - 2.0), rel=0, abs_tol=1e-5), out.item()


@test
def test_kl_narrowing_is_penalized():
    """Shrinking the cloud toward a point must cost something."""
    mu = torch.zeros(1, 1)
    wide = kl_divergence(mu, torch.tensor([[0.0]])).item()
    narrow = kl_divergence(mu, torch.tensor([[-4.0]])).item()
    assert narrow > wide, (wide, narrow)


@test
def test_kl_sums_over_latent_dimensions():
    """Four identical dimensions cost four times one dimension."""
    one = kl_divergence(torch.tensor([[1.0]]), torch.tensor([[0.0]])).item()
    four = kl_divergence(torch.ones(1, 4), torch.zeros(1, 4)).item()
    assert close(four, 4 * one, rel=0, abs_tol=1e-5), (one, four)


@test
def test_kl_gradients_flow():
    mu = torch.randn(BATCH, LATENT_DIM, requires_grad=True)
    logvar = torch.randn(BATCH, LATENT_DIM, requires_grad=True)
    kl_divergence(mu, logvar).sum().backward()
    assert mu.grad is not None and logvar.grad is not None


run("test_kl")

### Loss tests

Watch reduction: sum over pixels, stable across batch size.


In [ ]:
def _loss_parts():
    torch.manual_seed(0)
    return (
        torch.rand(BATCH, 1, 128, 128),   # x_hat
        torch.rand(BATCH, 1, 128, 128),   # x
        torch.randn(BATCH, LATENT_DIM),   # mu
        torch.randn(BATCH, LATENT_DIM),   # logvar
    )


@test
def test_loss_returns_three_scalars():
    for value in vae_loss(*_loss_parts()):
        assert value.dim() == 0, value.shape


@test
def test_loss_total_is_the_weighted_sum():
    total, recon, kl = vae_loss(*_loss_parts(), beta=2.5)
    assert close(total.item(), recon.item() + 2.5 * kl.item())


@test
def test_loss_beta_zero_drops_the_kl_term():
    total, recon, _ = vae_loss(*_loss_parts(), beta=0.0)
    assert close(total.item(), recon.item())


@test
def test_loss_perfect_reconstruction_is_free():
    """Holds for MSE. If you chose BCE, change this test and note why."""
    x = torch.rand(BATCH, 1, 128, 128)
    mu = torch.zeros(BATCH, LATENT_DIM)
    _, recon, _ = vae_loss(x, x, mu, torch.zeros_like(mu))
    assert close(recon.item(), 0.0, rel=0, abs_tol=1e-6), recon.item()


@test
def test_loss_sums_over_pixels():
    """The trap: a mean over 16384 pixels against a sum over latent dims.

    Two images with identical per-pixel error, one four times larger. A sum
    gives four times the cost. A mean gives the same cost for both.
    """
    mu = torch.zeros(1, LATENT_DIM)
    logvar = torch.zeros(1, LATENT_DIM)
    big = vae_loss(
        torch.full((1, 1, 128, 128), 0.1), torch.zeros(1, 1, 128, 128), mu, logvar
    )[1]
    small = vae_loss(
        torch.full((1, 1, 64, 64), 0.1), torch.zeros(1, 1, 64, 64), mu, logvar
    )[1]
    assert close(big.item(), 4 * small.item(), rel=1e-4), (big.item(), small.item())


@test
def test_loss_invariant_to_batch_size():
    """Doubling the batch with identical content must not change the loss.

    Catches summing over the batch where you meant to average.
    """
    torch.manual_seed(0)
    x = torch.rand(1, 1, 128, 128)
    x_hat = torch.rand(1, 1, 128, 128)
    mu = torch.randn(1, LATENT_DIM)
    logvar = torch.randn(1, LATENT_DIM)
    single = vae_loss(x_hat, x, mu, logvar)[0].item()
    doubled = vae_loss(
        x_hat.repeat(2, 1, 1, 1), x.repeat(2, 1, 1, 1),
        mu.repeat(2, 1), logvar.repeat(2, 1),
    )[0].item()
    assert close(doubled, single), (single, doubled)


@test
def test_loss_gradients_reach_the_model():
    model = ConvVAE(LATENT_DIM)
    x = torch.rand(BATCH, 1, 128, 128)
    x_hat, mu, logvar = model(x)
    vae_loss(x_hat, x, mu, logvar)[0].backward()
    assert any(
        p.grad is not None and p.grad.abs().sum() > 0 for p in model.parameters()
    )


run("test_loss")

### End-to-end

Overfit a tiny batch; loss should drop if pieces are wired correctly.


In [ ]:
@test
def test_end_to_end_loss_decreases():
    torch.manual_seed(0)
    model = ConvVAE(latent_dim=LATENT_DIM)
    x = torch.rand(BATCH, 1, 128, 128)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    first = None
    for step in range(30):
        x_hat, mu, logvar = model(x)
        total, _, _ = vae_loss(x_hat, x, mu, logvar, beta=1.0)
        opt.zero_grad()
        total.backward()
        opt.step()
        if step == 0:
            first = total.item()

    assert total.item() < first, (first, total.item())


run("test_end_to_end")

In [ ]:
# Everything for Exercise 3 at once. 35 tests.
run("test_construct") and run("test_encode") and run("test_reparam") \
    and run("test_decode") and run("test_forward") and run("test_kl") \
    and run("test_loss") and run("test_end_to_end")

## Training

Mirror Exercise 2 settings (opt/lr/epochs/split). Train at least two betas.


In [ ]:
from exercise3_vae import train_vae



In [ ]:
# Smoke test on one epoch before committing to a long run.
# vae, hist, best = train_vae(latent_dim=64, epochs=1, beta=1.0, tag="smoke")

# The sheet asks for at least two betas, so plan for two full runs:
# lo = train_vae(64, epochs=25, beta=0.01, tag="b001")
# hi = train_vae(64, epochs=25, beta=1.0,  tag="b1")

## Checks after training

Expect worse recon MSE than the AE. Save interpolation strip and prior samples.


In [ ]:
from exercise3_vae import interpolate, sample_from_prior



## Bonus 4 — frozen ResNet-18 control

Same split and k; change only the features. Not meant as a “better VAE”.


In [ ]:
from bonus_resnet import (
    build_feature_transform,
    build_frozen_resnet,
    extract_features,
    reduce_dimensions,
    evaluate_features,
)



### Preprocess tests


In [ ]:
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def _gray_image(value=128):
    return Image.fromarray(np.full((128, 128), value, dtype=np.uint8), mode="L")


@test
def test_transform_output_shape():
    assert build_feature_transform()(_gray_image()).shape == (3, 224, 224)


@test
def test_transform_size_is_configurable():
    assert build_feature_transform(size=128)(_gray_image()).shape == (3, 128, 128)


@test
def test_transform_replicates_grayscale_across_channels():
    """All three channels start from one source.

    They differ after normalization only because the statistics differ per
    channel. Undo the normalization and they must match.
    """
    out = build_feature_transform()(_gray_image())
    restored = torch.stack(
        [out[c] * IMAGENET_STD[c] + IMAGENET_MEAN[c] for c in range(3)]
    )
    assert torch.allclose(restored[0], restored[1], atol=1e-5)
    assert torch.allclose(restored[1], restored[2], atol=1e-5)


@test
def test_transform_applies_imagenet_normalization():
    """A pure white image must land at exactly (1 - mean) / std."""
    out = build_feature_transform()(_gray_image(255))
    for channel in range(3):
        expected = (1.0 - IMAGENET_MEAN[channel]) / IMAGENET_STD[channel]
        assert close(out[channel].mean(), expected, rel=0, abs_tol=1e-3), (
            channel, out[channel].mean().item(), expected
        )


run("test_transform")

### Frozen extractor tests

Needs pretrained weights (network access once).


In [ ]:
@test
def test_resnet_no_parameter_requires_gradient():
    assert all(not p.requires_grad for p in build_frozen_resnet().parameters())


@test
def test_resnet_is_in_evaluation_mode():
    """Batch normalization behaves differently in training mode."""
    assert not build_frozen_resnet().training


@test
def test_resnet_output_width_is_512():
    out = build_frozen_resnet()(torch.randn(2, 3, 224, 224))
    assert out.shape == (2, 512), out.shape


@test
def test_resnet_classifier_head_is_gone():
    """1000 outputs means the ImageNet head is still attached."""
    assert build_frozen_resnet()(torch.randn(1, 3, 224, 224)).shape[1] != 1000


@test
def test_resnet_weights_are_pretrained_not_random():
    """Two loads must agree. Random initialization would not.

    Catches passing weights=None and silently comparing your VAE against an
    untrained network.
    """
    x = torch.randn(2, 3, 224, 224)
    assert torch.allclose(build_frozen_resnet()(x), build_frozen_resnet()(x), atol=1e-5)


@test
def test_resnet_different_inputs_give_different_features():
    model = build_frozen_resnet()
    a = model(torch.zeros(1, 3, 224, 224))
    b = model(torch.ones(1, 3, 224, 224))
    assert not torch.allclose(a, b)


run("test_resnet")

### Extract / reduce / evaluate tests

Uses a tiny stub network offline where possible.


In [ ]:
from torch.utils.data import DataLoader, TensorDataset


class _Probe(nn.Module):
    """Deterministic stand-in for ResNet, so these tests need no download.

    Also records whether autograd was enabled while it ran.
    """

    def __init__(self):
        super().__init__()
        self.grad_was_enabled = None

    def forward(self, x):
        self.grad_was_enabled = torch.is_grad_enabled()
        return x.flatten(1)[:, :512]


def _stub_loader():
    images = torch.randn(10, 3, 224, 224)
    labels = torch.tensor([0, 1, 2, 3, 0, 1, 2, 3, 0, 1])
    return DataLoader(TensorDataset(images, labels), batch_size=4, shuffle=False)


@test
def test_extract_shapes_and_types():
    features, labels = extract_features(_Probe().eval(), _stub_loader())
    assert features.shape == (10, 512), features.shape
    assert labels.shape == (10,), labels.shape
    assert isinstance(features, np.ndarray) and isinstance(labels, np.ndarray)


@test
def test_extract_preserves_label_order():
    """An unshuffled loader must come back in dataset order.

    Shuffle here and features stop matching labels, which shows up as accuracy
    near chance with no obvious cause.
    """
    _, labels = extract_features(_Probe().eval(), _stub_loader())
    assert list(labels) == [0, 1, 2, 3, 0, 1, 2, 3, 0, 1], list(labels)


@test
def test_extract_builds_no_graph():
    """Extraction must run with autograd off.

    4000 images of retained activations will exhaust memory. Checking the flag
    before and after would prove nothing, since a stub that does nothing also
    leaves it unchanged. The probe reports the flag DURING the forward pass.
    """
    probe = _Probe().eval()
    extract_features(probe, _stub_loader())
    assert probe.grad_was_enabled is not None, "extractor was never called"
    assert probe.grad_was_enabled is False, "autograd was on during extraction"


@test
def test_reduce_output_shapes():
    tr, va = reduce_dimensions(
        np.random.randn(137, 512), np.random.randn(41, 512), n_components=64
    )
    assert tr.shape == (137, 64), tr.shape
    assert va.shape == (41, 64), va.shape


@test
def test_reduce_structure_survives():
    """Two well-separated clusters must stay separated."""
    rng = np.random.default_rng(0)
    stacked = np.vstack([rng.normal(0, 1, (100, 512)), rng.normal(20, 1, (100, 512))])
    reduced, _ = reduce_dimensions(stacked, stacked[:5], 8)
    gap = np.linalg.norm(reduced[:100].mean(0) - reduced[100:].mean(0))
    spread = reduced[:100].std(0).mean()
    assert gap > 5 * spread, (gap, spread)


@test
def test_reduce_shared_projection():
    """The same vector in both sets must land at the same coordinates.

    One projection fitted on train and applied to both sends a given vector to
    one point, whichever set it arrived in. Fitting each set separately gives
    two different bases, and the two copies land somewhere different.

    Shape checks cannot see this. Both halves come back the right size and each
    keeps its own internal structure. Only the comparison between them breaks,
    and it breaks silently: kNN accuracy drifts toward chance with nothing to
    debug.
    """
    rng = np.random.default_rng(1)
    train = rng.normal(size=(80, 512))
    val = np.vstack([train[3], rng.normal(size=(19, 512))])

    train_reduced, val_reduced = reduce_dimensions(train, val, n_components=16)
    assert np.allclose(train_reduced[3], val_reduced[0], atol=1e-6), (
        "row present in both sets landed at two different points: "
        "the projection was not shared"
    )


@test
def test_reduce_validation_is_not_refitted():
    """The projection is fitted on train only.

    Validation features go through that basis whatever they look like. Shift
    every validation vector by a large constant: under a shared projection the
    reduced validation points move together, keeping their relative geometry.
    Refit on the shifted set and the basis re-centres, erasing the shift.
    """
    rng = np.random.default_rng(2)
    train = rng.normal(size=(80, 512))
    val = rng.normal(size=(20, 512))

    _, plain = reduce_dimensions(train, val, n_components=16)
    _, shifted = reduce_dimensions(train, val + 50.0, n_components=16)

    moved = np.linalg.norm(shifted.mean(0) - plain.mean(0))
    assert moved > 1.0, (
        f"shifting the validation set moved its projection by only {moved:.4f}; "
        "the projection appears to be refitted on the validation data"
    )


def _separable():
    """Four clusters far enough apart that any sane classifier gets them."""
    rng = np.random.default_rng(0)
    centers = np.array([[0, 0], [30, 0], [0, 30], [30, 30]], dtype=float)
    tr_x, tr_y, va_x, va_y = [], [], [], []
    for label, center in enumerate(centers):
        tr_x.append(rng.normal(center, 1.0, (50, 2)))
        tr_y += [label] * 50
        va_x.append(rng.normal(center, 1.0, (20, 2)))
        va_y += [label] * 20
    return np.vstack(tr_x), np.array(tr_y), np.vstack(va_x), np.array(va_y)


@test
def test_evaluate_returns_expected_keys():
    assert set(evaluate_features(*_separable())) == {"accuracy", "silhouette", "report"}


@test
def test_evaluate_separable_data_scores_high():
    assert evaluate_features(*_separable())["accuracy"] > 0.95


@test
def test_evaluate_separable_data_has_high_silhouette():
    """Contrast with the near-zero silhouette your autoencoder codes scored."""
    assert evaluate_features(*_separable())["silhouette"] > 0.7


@test
def test_evaluate_random_features_score_near_chance():
    """Four balanced classes and no signal means roughly 0.25."""
    rng = np.random.default_rng(0)
    result = evaluate_features(
        rng.normal(size=(400, 16)), np.tile([0, 1, 2, 3], 100),
        rng.normal(size=(200, 16)), np.tile([0, 1, 2, 3], 50),
    )
    assert result["accuracy"] < 0.40, result["accuracy"]


@test
def test_evaluate_neighbor_count_is_honored():
    """Passing n_neighbors through must actually reach the classifier."""
    tr_x, tr_y, va_x, va_y = _separable()
    assert evaluate_features(tr_x, tr_y, va_x, va_y, n_neighbors=1)["accuracy"] > 0.9


run("test_extract") and run("test_reduce") and run("test_evaluate")

In [ ]:
# Everything, both parts. 57 tests.
run("test_")

## Checklist

- [ ] Exercise 2 baseline numbers recorded
- [ ] VAE tests green; two betas trained
- [ ] Interpolation + prior figures saved
- [ ] Frozen ResNet 5-NN / silhouette logged
- [ ] Short written answers filled in
